# Cold vs Hot Tumour Classification — Colab Runner

**BT3041 Term Paper — Group Project (6 members)**

This notebook runs the entire pipeline on Google Colab. Each team member owns one section. Run cells top-to-bottom the first time; after that each person can run only their section.

### Team assignments
| Member | Owns section |
|---|---|
| P1 | §1 Setup + §2 Data acquisition + §3 Preprocessing |
| P2 | §4 Immune scoring |
| P3 | §5 Dim-reduction + §6 Clustering |
| P4 | §7 Hypothesis testing |
| P5 | §8 Classification |
| P6 | §9 Survival + §10 External validation |
| ALL | §11 Final figures |

---
## §1 Setup (run once, takes ~3 min)

**Before running this notebook:**
1. Upload `cold_hot_tumour_project.zip` to your Google Drive root
2. Share the Drive folder with all team members (so everyone sees the outputs)
3. In Colab: Runtime → Change runtime type → None (or GPU if you want speed)

In [ ]:
# 1.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.2 Unzip the project into Colab workspace
import os, shutil, zipfile

# ---- EDIT THIS if you named your Drive folder differently ----
DRIVE_FOLDER = '/content/drive/MyDrive/BT3041_project'
# --------------------------------------------------------------

ZIP_PATH = f'{DRIVE_FOLDER}/cold_hot_tumour_project.zip'
WORK     = '/content/cold_hot_tumour_project'

assert os.path.exists(ZIP_PATH), f'Upload the zip to {ZIP_PATH} first!'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content/')

os.chdir(WORK)
print('Working dir:', os.getcwd())
!ls

In [ ]:
# 1.3 Install dependencies (Colab already has numpy/pandas/sklearn/matplotlib)
!pip install -q gseapy GEOparse lifelines umap-learn scikit-posthocs mygene xgboost

In [ ]:
# 1.4 Persist all data + outputs to Drive via symlinks
#     (so `!python script.py` subprocesses save directly to Drive and
#      nothing is lost when the Colab VM recycles)
import os, shutil
from pathlib import Path

SHARED = Path(DRIVE_FOLDER) / 'cold_hot_tumour_outputs'
for sub in ('data/raw', 'data/processed',
            'outputs/figures', 'outputs/tables', 'outputs/models'):
    (SHARED / sub).mkdir(parents=True, exist_ok=True)

# Replace local folders with symlinks -> Drive
def _relink(local: Path, target: Path) -> None:
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        # move anything accidentally written locally into Drive first
        for f in local.rglob('*'):
            if f.is_file():
                rel = f.relative_to(local)
                dst = target / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists():
                    shutil.copy2(f, dst)
        shutil.rmtree(local)
    local.symlink_to(target)

project = Path('/content/cold_hot_tumour_project')
_relink(project / 'data',    SHARED / 'data')
_relink(project / 'outputs', SHARED / 'outputs')

print('Project folders linked to Drive:')
!ls -la {project}/ | grep -E 'data|outputs'
print('\nDrive storage root:', SHARED)
print('Anything your scripts save will persist across Colab restarts.')

---
## §2 Data acquisition — OWNER: P1

⏱  ~30-60 min  |  ~3 GB downloaded  |  This is the slowest step, run it once and let the team reuse the output.

Downloads:
- TCGA-COAD (456 samples)  from NIH GDC API
- TCGA-READ (163 samples)  from NIH GDC API
- GSE39582  (566 samples)  from NCBI GEO
- GSE17538  (244 samples)  from NCBI GEO

If the Colab runtime disconnects, re-run this cell — the script skips files that already downloaded.

In [ ]:
!python 01_data_acquisition.py

---
## §3 Preprocessing — OWNER: P1

Log2 transform, low-gene filter, QQ / Shapiro normality checks, top-5000 variance subset.

In [ ]:
!python 02_preprocessing.py

---
## §4 Immune scoring — OWNER: P2

ssGSEA + ESTIMATE → Hot/Cold/Intermediate labels (tertile split on immune score).

⏱  ~15-30 min on 1,400 samples.

In [ ]:
!python 03_immune_scoring.py

---
## §5 Dimensionality reduction — OWNER: P3

PCA + ICA + MDS + t-SNE + UMAP — all five methods. Produces labelled scatter plots.

In [ ]:
!python 04_dimensionality_reduction.py

---
## §6 Clustering — OWNER: P3

k-means + hierarchical (Ward) + DBSCAN. Silhouette sweep. ARI vs Hot/Cold labels.

In [ ]:
!python 05_clustering.py

---
## §7 Hypothesis testing — OWNER: P4

t-test + ANOVA + Kruskal + F-test of variance + χ² on clinical vars + Benjamini-Hochberg FDR + Bonferroni. Volcano plot + top-40 heatmap.

In [ ]:
!python 06_hypothesis_testing.py

---
## §8 Classification — OWNER: P5

kNN + SVM(linear,RBF) + LogReg + RF + XGB. 5-fold stratified CV. ROC + confusion matrices.

In [ ]:
!python 07_classification.py

---
## §9 Survival analysis — OWNER: P6

Kaplan-Meier + log-rank + Cox PH + PLS regression.

In [ ]:
!python 08_survival_analysis.py

---
## §10 External validation — OWNER: P6

Train on TCGA → predict on GSE39582. The single biggest point-scorer vs the previous-year project.

In [ ]:
!python 09_validation_external.py

---
## §11 Final composite figures — OWNER: whole team

Stitches all plots into report-ready panels in `outputs/figures/report/`.

In [ ]:
!python 10_final_report_figures.py

---
## §12 Quick inspection of outputs

Use these cells to glance at what the pipeline produced.

In [ ]:
import pandas as pd

print('--- Label distribution per cohort ---')
for f in config.TABLES_DIR.glob('labels_*.csv'):
    d = pd.read_csv(f)
    print(f.stem, d['label'].value_counts().to_dict())

print('\n--- Classifier CV results ---')
for f in config.TABLES_DIR.glob('cv_results_*.csv'):
    print(f.stem); print(pd.read_csv(f).to_string(index=False)); print()

print('\n--- External validation ---')
for f in config.TABLES_DIR.glob('external_validation_summary_*.csv'):
    print(pd.read_csv(f).to_string(index=False))

In [ ]:
# Show a few key figures inline
from IPython.display import Image, display
for name in ['report/Fig2_embedding_panel.png',
             'report/Fig3_survival_panel.png',
             'report/Fig4_model_comparison.png']:
    p = config.FIGURES_DIR / name
    if p.exists():
        print(name)
        display(Image(str(p)))